This notebook converts the preprocessed SMS messages into numerical TF-IDF feature vectors for training the Support Vector Machine (SVM) model.

In [1]:
import os
import sys
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

print("Python version:", sys.version)
print("Libraries imported successfully.")

Python version: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Libraries imported successfully.


In [2]:
data_path = "../data/preprocessed_sms_spam.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully.
Dataset shape: (5044, 11)


,label,message,character_count,word_count,sentence_count,original_message,clean_message,clean_char_length,clean_word_count,gru_message,label_encoded
0,ham,"Go until jurong point, crazy.. Available only ...",111,24,2,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis great wo...,78,14,go until jurong point crazy available only in ...,0
1,ham,Ok lar... Joking wif u oni...,29,8,2,Ok lar... Joking wif u oni...,ok lar joking wif oni,21,5,ok lar joking wif u oni,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,37,2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...,99,20,free entry in a wkly comp to win fa cup final ...,1
3,ham,U dun say so early hor... U c already then say...,49,13,1,U dun say so early hor... U c already then say...,dun say early hor already say,29,6,u dun say so early hor u c already then say,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,15,1,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though,35,7,nah i don t think he goes to usf he lives arou...,0


In [3]:
print("Current working directory:")
print(os.getcwd())

Current working directory:
d:\SMS_Spam_Detection\notebooks


In [4]:
print("Dataset columns:")
print(df.columns.tolist())

Dataset columns:
['label', 'message', 'character_count', 'word_count', 'sentence_count', 'original_message', 'clean_message', 'clean_char_length', 'clean_word_count', 'gru_message', 'label_encoded']


In [6]:
print(df.columns.tolist())

['label', 'message', 'character_count', 'word_count', 'sentence_count', 'original_message', 'clean_message', 'clean_char_length', 'clean_word_count', 'gru_message', 'label_encoded']


In [9]:
text_column = "cleaned_message_svm"
target_column = "label_encoded"

In [10]:
text_column = "clean_message"
target_column = "label_encoded"

In [11]:
text_column = "clean_message"
target_column = "label_encoded"

X = df[text_column].fillna("")
y = df[target_column]

print("Number of messages:", len(X))
print("\nTarget distribution:")
print(y.value_counts())

Number of messages: 5044

Target distribution:
label_encoded
0    4465
1     579
Name: count, dtype: int64


In [12]:
empty_message_count = X.str.strip().eq("").sum()

print("Empty SVM messages:", empty_message_count)

Empty SVM messages: 0


In [13]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training messages:", len(X_train_text))
print("Testing messages:", len(X_test_text))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training messages: 4035
Testing messages: 1009

Training class distribution:
label_encoded
0    3572
1     463
Name: count, dtype: int64

Testing class distribution:
label_encoded
0    893
1    116
Name: count, dtype: int64


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print(tfidf_vectorizer)

TfidfVectorizer(max_df=0.95, max_features=5000, min_df=2, ngram_range=(1, 2),
                sublinear_tf=True)


In [15]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print("TF-IDF transformation completed.")
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

TF-IDF transformation completed.
Training TF-IDF shape: (4035, 5000)
Testing TF-IDF shape: (1009, 5000)


In [16]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Number of features:", len(feature_names))
print("\nFirst 20 features:")
print(feature_names[:20])

Number of features: 5000

First 20 features:
['aathi' 'abi' 'abiola' 'abj' 'able' 'able get' 'able pay' 'abt' 'abt tht'
 'abta' 'abta complimentary' 'aburo' 'aburo enjoy' 'ac' 'acc' 'accept'
 'access' 'accidentally' 'account' 'account detail']


In [17]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

vectorizer_path = "../models/tfidf_vectorizer.pkl"

joblib.dump(tfidf_vectorizer, vectorizer_path)

print("TF-IDF vectorizer saved successfully.")
print("Saved location:", vectorizer_path)

TF-IDF vectorizer saved successfully.
Saved location: ../models/tfidf_vectorizer.pkl


In [18]:
train_data = pd.DataFrame({
    "clean_message": X_train_text,
    "label_encoded": y_train
}).reset_index(drop=True)

test_data = pd.DataFrame({
    "clean_message": X_test_text,
    "label_encoded": y_test
}).reset_index(drop=True)

train_data.to_csv("../data/svm_train_data.csv", index=False)
test_data.to_csv("../data/svm_test_data.csv", index=False)

print("Training data saved:", train_data.shape)
print("Testing data saved:", test_data.shape)

Training data saved: (4035, 2)
Testing data saved: (1009, 2)


In [20]:
print("TF-IDF FEATURE ENGINEERING SUMMARY")
print("=" * 50)

print("Training messages:", len(X_train_text))
print("Testing messages:", len(X_test_text))
print("Number of TF-IDF features:", X_train_tfidf.shape[1])
print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)
print("Vectorizer saved successfully.")

print("\nFeature engineering is complete.")
print("Next step: Train the SVM model.")

TF-IDF FEATURE ENGINEERING SUMMARY
Training messages: 4035
Testing messages: 1009
Number of TF-IDF features: 5000
Training matrix shape: (4035, 5000)
Testing matrix shape: (1009, 5000)
Vectorizer saved successfully.

Feature engineering is complete.
Next step: Train the SVM model.
